# Import Needed Methods

In [1]:
from cooccurence_network import *
from scrap_issue_content import *
from scrap_issue_links import *

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/zhangmengqi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/zhangmengqi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/zhangmengqi/nltk_data...


# Get Links for Open Issues and Closed Issues
The links will be saved as .csv files under the provided path \
the follwing cell will run test on a small scale sample

In [8]:
with sync_playwright() as playwright:

    test_link = "https://github.com/pytorch/pytorch/issues?page=495&q=is%3Aissue+is%3Aopen"
    test_directory = "./scrapper_results/test_results"

    scrap_link(test_link, test_directory, playwright, 0, False)

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

the following cell will run the script on full scale

In [ ]:
with sync_playwright() as playwright:
    results_directory_open = './scrapper_results/open_issues'
    link_open = "https://github.com/pytorch/pytorch/issues?q=is%3Aopen+is%3Aissue" # link to open issues

    results_directory_closed = './scrapper_results/closed_issues'
    link_closed = "https://github.com/pytorch/pytorch/issues?q=is%3Aissue+is%3Aclosed" # link to closed issues

    test_link = "https://github.com/pytorch/pytorch/issues?page=495&q=is%3Aissue+is%3Aopen"
    test_directory = "./scrapper_results/test_results"

    scrap_link(link_open, results_directory_open, playwright, 2)
    scrap_link(link_closed, results_directory_closed, playwright, 2)

    # scrap_link(test_link, test_directory, playwright, 0, False)

# Get Issue Content for Each Issues Link

the following cell is for testing the functionality on a sample list of link before run the script on full scale

In [ ]:
test_issues_links = "./scrapper_results/test_results/links.csv"

# test all functions
test_issues_contents = scrap_issues_content(test_issues_links, result_directory="./scrapper_results/test_results")
issues_contents = load_saved_results("./scrapper_results/test_results/issues_contents.pkl")
if len(issues_contents) > 0: 
    issues_contents[0].generate_file_output("./scrapper_results/test_results/test_file_recovery.txt")

test scrap_single_issue()
test_issue = scrap_single_issue("https://github.com/pytorch/pytorch/issues/113208")
test_issue.generate_file_output("./scrapper_results/test_results/issues_contents/issue_1.txt")

the following cell will run the script on the full scale

In [ ]:
open_issues_links = "./scrapper_results/open_issues/links.csv"
closed_issues_links = "./scrapper_results/closed_issues/links.csv"

print("scrapping open issues")
open_issues_contents = scrap_issues_content(open_issues_links, result_directory="./scrapper_results/open_issues", maximum_wait_time=0)

print("scrapping closed issues")
closed_issues_contents = scrap_issues_content(closed_issues_links, result_directory="./scrapper_results/closed_issues", maximum_wait_time=0)

# Co-Occurence Network
First prepare the issue contents using the saved results from both opened issues and closed issues

In [2]:
# prepare the issue contents
pickle_path_open = "./scrapper_results/open_issues/issues_contents.pkl"
pickle_path_closed = "./scrapper_results/closed_issues/issues_contents.pkl"
output_path = "./scrapper_results"

issues_contents_open = load_saved_results(pickle_path_open)
issues_contents_closed = load_saved_results(pickle_path_closed)

issue_contents = issues_contents_open + issues_contents_closed

Then, create the Co-Occurence network, and save results into pickle files for future reuse

In [3]:
# udpate the cooccurence network
coocurence_pairs, unique_words_total_occurence, unique_words_pair_count, report_str, cooccurence_pairs_origin_issue_index = cooccurence_network(issue_contents, maximum_word_distance=2)

# save the variables to checkpoint
save_variables_to_checkpoint(output_path, 
                                cooccurence_pairs=coocurence_pairs, 
                                unique_words_total_occurence=unique_words_total_occurence, 
                                unique_words_pair_count=unique_words_pair_count,
                                pairs_origin_issue_index=cooccurence_pairs_origin_issue_index)
save_report_to_file(output_path, report_str)





------------------------getting cooccurence pairs-------------------------


extracting first comments...


100%|██████████| 38019/38019 [00:28<00:00, 1312.22it/s]



extracting cooccurence pairs...


100%|██████████| 38019/38019 [00:09<00:00, 3995.82it/s]



calculating pair counts of each word...


100%|██████████| 1622996/1622996 [00:00<00:00, 3025436.77it/s]




----------------------------finished---------------------------------



------------------------getting cooccurence pairs-------------------------

the most occured word pair is: 
frozenset({'python', 'version'}), occured 14411 times

the word has the most total occurence is: version, occured 282569 times in 10532 pairs
the word occured in the most pairs is: line, occured 101263 times in 12182 pairs
the average number of pairs each word has is: 21.503
the average number of occurence each word has is: 82.955
the average number of occurence each pair has is: 3.858


----------------------------finished---------------------------------


saving variables to ./scrapper_results...
saved

saving report to file...
saved


Now, load the Co-Occurence network and its relevant data from saved pickle files, and generate an html file for presenting the network

In [4]:
# load the variables from checkpoint
cooccurence_pairs, unique_words_total_occurence, unique_words_pair_count, cooccurence_pairs_origin_issue_index = load_variables_from_checkpoint(output_path)

# get the origin issues of each pair
cooccurence_pairs_origin_issues_issue_contents = get_cooccurence_pairs_origin_issues(cooccurence_pairs_origin_issue_index, issue_contents)

# generate interactive cooccurence network
plots_storage = plots(cooccurence_pairs, unique_words_total_occurence, unique_words_pair_count)
plots_storage.plot_interactive_cooccurence_network(dir_path=output_path, number_of_pairs=2000, height="1250px", width="2500px")


loading variables from ./scrapper_results...
loaded

getting origin issues of each pair...


100%|██████████| 1622996/1622996 [00:03<00:00, 472041.39it/s]




---------------plotting interactive cooccurence network--------------

generating html file...


100%|██████████| 2000/2000 [00:00<00:00, 10549.74it/s]

number of nodes: 820 
number of edges: 2000



------------------------------finished------------------------------



We are able to spot some clusters from the Co-Occurence network, by useing the provided functions, we are able to trace back to the original comments that gives us the keywords in those clusters \
By examine a sample of the resutls, we can confirm the actual topic of each clusters

In [5]:
# get sample of comments for pairs
pairs = [['version', 'runtime']]
sample_size = 10
for pair in pairs:
    sample = get_comment_sample_for_pair(pair, 
                                         cooccurence_pairs_origin_issues_issue_contents, 
                                         sample_size=sample_size, 
                                         dir_path=output_path+"/sample_comments_for_pairs")


getting sample of comments for pair...
finished


# define a threshold for count clusters in the co-occurence network

In [1]:
from cooccurence_network import *

# prepare the issues contents
pickle_path_open = "./scrapper_results/open_issues/issues_contents.pkl"
pickle_path_closed = "./scrapper_results/closed_issues/issues_contents.pkl"
output_path = "./scrapper_results/cooccurence_network_results"

issues_contents_open = load_saved_results(pickle_path_open)
issues_contents_closed = load_saved_results(pickle_path_closed)

issue_contents = issues_contents_open + issues_contents_closed

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/zhangmengqi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/zhangmengqi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/zhangmengqi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [20]:
# udpate the cooccurence network
cooccurence_pairs, unique_words_total_occurence, unique_words_pair_count, report_str, cooccurence_pairs_origin_issue_index = cooccurence_network(issue_contents, 
                                                                                                                                                maximum_word_distance=2, 
                                                                                                                                                stem=False, 
                                                                                                                                                lemmatize=True)



------------------------getting cooccurence pairs-------------------------


extracting first comments...


100%|██████████| 38019/38019 [00:27<00:00, 1379.99it/s]



extracting cooccurence pairs...


100%|██████████| 38019/38019 [00:08<00:00, 4636.27it/s]



calculating pair counts of each word...


100%|██████████| 1622996/1622996 [00:00<00:00, 2986809.29it/s]




----------------------------finished---------------------------------



------------------------getting cooccurence pairs-------------------------

the most occured word pair is: 
frozenset({'version', 'python'}), occured 14411 times

the word has the most total occurence is: version, occured 282569 times in 10532 pairs
the word occured in the most pairs is: line, occured 101263 times in 12182 pairs
the average number of pairs each word has is: 21.503
the average number of occurence each word has is: 82.955
the average number of occurence each pair has is: 3.858


----------------------------finished---------------------------------



New create a histogram for dictionary unique_words_total_occurence to see the distribution of number of pairs for each word.

see if there is a significant drop that can be used as the threshold of cluster

In [43]:
plots_storage = plots(cooccurence_pairs, unique_words_total_occurence, unique_words_pair_count)
showed_word_pair_count = plots_storage.get_word_pair_count_in_showed_network(number_of_clusters=2000)

# transform the showed word pair count to a dataframe
showed_word_pair_count_df = pd.DataFrame(showed_word_pair_count, columns=["word_pair", "count"])
showed_word_pair_count_df = showed_word_pair_count_df.sort_values(by="count", ascending=False)

showed_word_pair_count_df.head()






---------------getting clusters from cooccurence network--------------


100%|██████████| 2000/2000 [00:00<00:00, 86028.18it/s]

                            count
(version, python)           14411
(version, collect)          12937
(cuda, version)             12758
(runtime, version)          12288
(build, used)               11415
...                           ...
(detached, distribution)        1
(explicitly, distribution)      1
(constructing, resorting)       1
(alternative, resorting)        1
(b, element-wise)               1

[1622996 rows x 1 columns]
--------------------
2000
--------------------
                            count
(version, python)           14411
(version, collect)          12937
(cuda, version)             12758
(runtime, version)          12288
(build, used)               11415
...                           ...
(cc, jeffdaily)               268
(result, error)               268
(yanboliang, voznesenskym)    268
(external, unresolved)        268
(tensor, index)               268

[2000 rows x 1 columns]


,word_pair,count


In [44]:
plots_storage.cooccurence_pairs

,count
"(version, python)",14411
"(version, collect)",12937
"(cuda, version)",12758
"(runtime, version)",12288
"(build, used)",11415
...,...
"(detached, distribution)",1
"(explicitly, distribution)",1
"(constructing, resorting)",1
"(alternative, resorting)",1
